In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch07-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "07-01-murcko-scaffolds.py"

# 在原分子上標出 Murcko 骨架

第一次執行需下載 ESOL。圖中保留原分子的全部原子，以底色標出留下的骨架。要做小練習，可把程式接在下一格的末端，再執行最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os

import deepchem as dc
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Scaffolds import MurckoScaffold

# MoleculeNet ESOL，1,128 筆實測 log S（S 以 mol/L 表示），DOI 10.1021/ci034243x
_, (data,), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None, transformers=[],
    reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
records = [(Chem.MolToSmiles(Chem.MolFromSmiles(s)), y)
           for s, y in zip(data.ids, data.y[:, 0])]
table = pd.DataFrame(records, columns=["smiles", "logS"]).groupby("smiles").median()
table["mol"] = [Chem.MolFromSmiles(s) for s in table.index]
table["scaffold"] = [
    MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    for mol in table["mol"]
]
counts = table["scaffold"].value_counts()
print("Records / structures:", len(data), len(table))
print("Ring frameworks / acyclic structures:", len(counts.drop("")), counts[""])
print(counts.drop("").head(3).to_string())
benzene = Chem.MolFromSmiles("c1ccccc1")
print("Structures containing benzene:", sum(m.HasSubstructMatch(benzene) for m in table.mol))

examples = {"Phenetole": "CCOc1ccccc1", "Diphenyl ether": "c1ccc(Oc2ccccc2)cc1",
            "Phenol": "Oc1ccccc1", "1-Butanol": "CCCCO"}
mols, highlights, labels = [], [], []
for name, smiles in examples.items():
    record = table.loc[Chem.MolToSmiles(Chem.MolFromSmiles(smiles))]
    mol = record["mol"]
    core = MurckoScaffold.GetScaffoldForMol(mol)
    atoms = mol.GetSubstructMatch(core) if core.GetNumAtoms() else ()
    mols.append(mol)
    highlights.append(list(atoms))
    labels.append(name)
    print(name, record["scaffold"] or "(acyclic)", f"log S = {record['logS']:.2f}")
grid = Draw.MolsToGridImage(mols, molsPerRow=2, subImgSize=(360, 230),
                            legends=labels, highlightAtomLists=highlights)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(exist_ok=True)
grid.save(figure_dir / "07-01-murcko-scaffolds.png")


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "07-01-murcko-scaffolds.png")))